# Practical 10 — Bank Customer Churn Prediction using a Neural Network

Standalone solution notebook for **ML Practical 10**. Run cells from top to bottom and place the required dataset CSV in the working directory.https://www.kaggle.com/datasets/barelydedicated/bank-customer-churn-modeling?utm_source=chatgpt.com

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, mean_squared_error, mean_absolute_error,
    r2_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

---
# Practical 10 — Bank Customer Churn Prediction using a Neural Network
**Aim:** Predict whether a bank customer will leave (`Exited`/churn = 1) using a neural-network classifier. Split the data, normalize numeric features, and print accuracy and confusion matrix.

Expected dataset: `Churn_Modelling.csv` (or similar), with target `Exited` and columns such as `CustomerId`, `Surname`, `CreditScore`, `Geography`, `Gender`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, and `EstimatedSalary`.

In [ ]:
BANK_PATH = "Churn_Modelling.csv"
if not os.path.exists(BANK_PATH):
    alternatives = ["bank.csv", "bank_churn.csv", "Bank_Customer_Churn.csv"]
    found = next((f for f in alternatives if os.path.exists(f)), None)
    if found:
        BANK_PATH = found
    else:
        raise FileNotFoundError(
            "Bank churn dataset not found. Put Churn_Modelling.csv beside this notebook and set BANK_PATH."
        )

bank = pd.read_csv(BANK_PATH)
print("Shape:", bank.shape)
display(bank.head())

In [ ]:
target_candidates = ["exited", "churn", "left", "attrition"]
bank_col_map = {str(c).strip().lower(): c for c in bank.columns}
TARGET_COL = next((bank_col_map[c] for c in target_candidates if c in bank_col_map), None)
if TARGET_COL is None:
    raise ValueError("Could not identify target column. Set TARGET_COL to the churn target column.")

# Remove identifier-like columns; retain useful numeric and categorical predictors.
drop_cols = [c for c in bank.columns if str(c).strip().lower() in
             ["rownumber", "customerid", "surname", "id"]]
X = bank.drop(columns=[TARGET_COL] + drop_cols, errors="ignore")
y = pd.to_numeric(bank[TARGET_COL], errors="coerce")
valid = y.notna()
X, y = X.loc[valid], y.loc[valid].astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse one-hot output to dense arrays for sklearn MLPClassifier.
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()
print("Training matrix:", X_train_processed.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    random_state=RANDOM_STATE
)
nn_model.fit(X_train_processed, y_train)
y_pred = nn_model.predict(X_test_processed)

print("Neural Network Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Bank Churn — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**Improvement ideas:** try different hidden-layer sizes, regularization (`alpha`), class weights/resampling for imbalanced labels, and cross-validation. This solution uses scikit-learn's `MLPClassifier`, a feed-forward neural network, so TensorFlow is not required.